# LAVEE — 02 · Saved-model detection, tracking and dashboard

**No training in this notebook.** Load the two detector families and the corresponding five hybrid
classifiers already installed in the project. Choose YOLOv8s or YOLO12n, then the classification method.
YOLO generates boxes and ByteTrack IDs; the selected classifier labels each detected crop.

The supplied **LAVEE_Dashboard_Colab_Bundle.zip** contains the current source code, two model families,
real research results and demos. It excludes previous uploads, generated recordings, logs and temporary credentials.
Upload it once to Drive (recommended) or select it directly when prompted below.

YOLO12 uses the **Fold 1 demo export**. Its five heads all consume 16 × 512 frozen ResNet18 spatial tokens.
YOLOv8s uses the older final classifiers and their original preprocessing. Never mix their weights or results.
The actual image/video tests passed locally for all 12 combinations. Colab execution and physical webcam
access have not been verified in this environment; GPU availability and browser proxy behavior can differ.


## 1. Install runtime dependencies
Use a fresh Colab Python runtime. GPU is optional; the current crop-classifier code runs on CPU.


In [ ]:
%pip -q install ultralytics==8.4.149 "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "python-multipart>=0.0.18" "opencv-python>=4.10" "imageio-ffmpeg>=0.5" "lap>=0.5.12" "websockets>=12" "timm>=1.0,<2" torchvision requests pandas
import os,sys,json,time,zipfile,hashlib,base64,uuid,subprocess
from pathlib import Path
import torch
print('PyTorch:',torch.__version__,'CUDA available:',torch.cuda.is_available())


## 2. Load the dashboard bundle

Set `BUNDLE_PATH` to your Drive ZIP location. Leave it empty to upload from your computer.
Extraction is restricted to `/content/lavee_project`. The bundle checksum manifest is checked before model loading.


In [ ]:
USE_DRIVE=True
BUNDLE_PATH='/content/drive/MyDrive/LAVEE_Dashboard_Colab_Bundle.zip' # EDIT, or '' to upload
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if not BUNDLE_PATH or not Path(BUNDLE_PATH).is_file():
    print('Select LAVEE_Dashboard_Colab_Bundle.zip')
    from google.colab import files
    uploaded=files.upload()
    choices=[Path(n) for n in uploaded if n.lower().endswith('.zip')]
    assert len(choices)==1,'Select exactly one dashboard bundle ZIP.'
    BUNDLE_PATH=str(choices[0])
    del uploaded
PROJECT=Path('/content/lavee_project')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as z:
    for entry in z.infolist():
        target=(PROJECT/entry.filename).resolve()
        assert target.is_relative_to(PROJECT.resolve()),'Unsafe ZIP path.'
        assert (entry.external_attr>>16)&0o170000!=0o120000,'Symlinks are not supported.'
    z.extractall(PROJECT)
DASH=PROJECT/'dashboard'
manifest=json.loads((PROJECT/'bundle_manifest.json').read_text())
for name,expected in manifest['sha256'].items():
    path=PROJECT/name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==expected,name+' failed integrity check'
print('Verified',len(manifest['sha256']),'files.')
os.chdir(DASH)
if str(DASH) not in sys.path:sys.path.insert(0,str(DASH))


## 3. Project source code used by the dashboard

These cells contain the actual project modules, not pseudocode. They write the same source included
in the bundle. Run them before importing the application. The ZIP supplies models, statistics and demos.


### recurrent_models.py


In [ ]:
%%writefile /content/lavee_project/dashboard/recurrent_models.py
"""Shared recurrent heads; checkpoints must specify input basis and six-class order."""
import torch
from torch import nn

class RecurrentClassifier(nn.Module):
    def __init__(self, input_dim, bidirectional=False, attention=False, hidden_dim=64, num_classes=6):
        super().__init__()
        self.rnn=nn.LSTM(input_dim,hidden_dim,num_layers=2,batch_first=True,
                         bidirectional=bidirectional,dropout=.2)
        width=hidden_dim*(2 if bidirectional else 1)
        self.attention=nn.Sequential(nn.Linear(width,64),nn.Tanh(),nn.Linear(64,1)) if attention else None
        self.head=nn.Sequential(nn.LayerNorm(width),nn.Linear(width,32),nn.ReLU(),nn.Dropout(.2),nn.Linear(32,num_classes))

    def forward(self,x):
        sequence,_=self.rnn(x)
        pooled=(sequence*self.attention(sequence).softmax(1)).sum(1) if self.attention is not None else sequence[:,-1]
        return self.head(pooled)


### yolo12_heads.py


In [ ]:
%%writefile /content/lavee_project/dashboard/yolo12_heads.py
from torch import nn
import torch

class CNNHead(nn.Module):
    def __init__(self):
        super().__init__();self.conv=nn.Sequential(nn.Conv1d(512,128,3,padding=1),nn.ReLU(),nn.Conv1d(128,64,3,padding=1),nn.ReLU());self.fc=nn.Sequential(nn.LayerNorm(64),nn.Linear(64,6))
    def forward(self,x):return self.fc(self.conv(x.transpose(1,2)).mean(-1))

class LSTMHead(nn.Module):
    def __init__(self,bi=False,attention=False):
        super().__init__();self.rnn=nn.LSTM(512,64,2,batch_first=True,bidirectional=bi,dropout=.2)
        width=128 if bi else 64
        self.att=nn.Sequential(nn.Linear(width,64),nn.Tanh(),nn.Linear(64,1)) if attention else None
        self.fc=nn.Sequential(nn.LayerNorm(width),nn.Linear(width,32),nn.ReLU(),nn.Dropout(.2),nn.Linear(32,6))
    def forward(self,x):
        z,_=self.rnn(x)
        z=(z*self.att(z).softmax(dim=1)).sum(1) if self.att else z[:,-1]
        return self.fc(z)

class TransformerHead(nn.Module):
    def __init__(self):
        super().__init__();self.embed=nn.Linear(512,128);self.pos=nn.Parameter(torch.zeros(1,16,128))
        self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(128,4,dim_feedforward=256,dropout=.1,batch_first=True),num_layers=2,enable_nested_tensor=False)
        self.fc=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,6))
    def forward(self,x):return self.fc(self.encoder(self.embed(x)+self.pos).mean(1))

METHODS=['CNN','LSTM','Transformer','BiLSTM_Attention','cGAN_Transformer']
def make_head(name):
    if name=='CNN':return CNNHead()
    if name=='LSTM':return LSTMHead()
    if name=='BiLSTM_Attention':return LSTMHead(bi=True,attention=True)
    if name in ('Transformer','cGAN_Transformer'):return TransformerHead()
    raise ValueError(name)


### yolo12_backend.py


In [ ]:
%%writefile /content/lavee_project/dashboard/yolo12_backend.py
"""Runtime matching the deadline Colab notebook's Fold 1 export."""
from pathlib import Path
import csv
import json
import math
import statistics
import threading
import cv2
import torch
import timm
from torch import nn
from PIL import Image
from torchvision import transforms
from yolo12_heads import make_head

ROOT = Path(__file__).resolve().parent / 'models/yolo12_fold1'
FILES = {'cnn':'CNN_head.pt', 'lstm':'LSTM_head.pt', 'transformer':'Transformer_head.pt',
         'bilstm_attention':'BiLSTM_Attention_head.pt', 'cgan_transformer':'cGAN_Transformer_head.pt'}
METHODS = {'cnn':'CNN', 'lstm':'LSTM', 'transformer':'Transformer',
           'bilstm_attention':'BiLSTM_Attention', 'cgan_transformer':'cGAN_Transformer'}
REQUIRED = ['yolo12_best.pt', 'resnet18_frozen.pt', 'run_config.json', 'cv_progress.json', 'cv_summary.csv', *FILES.values()]
TF = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
                         transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
CACHE = {}
LOCK = threading.Lock()

def ready():
    return (ROOT/'validated.json').is_file() and all((ROOT/f).is_file() and (ROOT/f).stat().st_size for f in REQUIRED)

def config():
    return json.loads((ROOT/'run_config.json').read_text())

def names():
    return ['drone' if n.lower()=='uav' else n.lower() for n in config()['names']]

def load(technique):
    if 'extractor' not in CACHE:
        ex = timm.create_model('resnet18', pretrained=False, num_classes=0)
        ex.load_state_dict(torch.load(ROOT/'resnet18_frozen.pt', map_location='cpu', weights_only=True), strict=True)
        CACHE['extractor'] = ex.eval()
    if technique not in CACHE:
        head = make_head(METHODS[technique])
        head.load_state_dict(torch.load(ROOT/FILES[technique], map_location='cpu', weights_only=True), strict=True)
        CACHE[technique] = head.eval()
    return CACHE['extractor'], CACHE[technique], names()

def classify(image, boxes, technique):
    if not boxes:return []
    h,w = image.shape[:2]
    crops = []
    for x1,y1,x2,y2 in boxes:
        # Same tight boxes and resize as notebook inference; no old 8% context/padding.
        l,t,r,b = max(0,int(round(x1))),max(0,int(round(y1))),min(w,int(round(x2))),min(h,int(round(y2)))
        if r<=l or b<=t:raise ValueError('Invalid detector crop.')
        crops.append(TF(Image.fromarray(cv2.cvtColor(image[t:b,l:r],cv2.COLOR_BGR2RGB))))
    with LOCK, torch.inference_mode():
        ex,head,classes = load(technique)
        tokens = nn.functional.adaptive_avg_pool2d(ex.forward_features(torch.stack(crops)),(4,4)).flatten(2).transpose(1,2)
        probs = head(tokens).softmax(1)
    return [{'class_id':int(p.argmax()),'class_name':classes[int(p.argmax())], 'confidence':round(float(p.max()),4)} for p in probs]

def research_summary():
    folds = json.loads((ROOT/'cv_progress.json').read_text())
    rows = list(csv.DictReader((ROOT/'cv_summary.csv').open()))
    detector = {}
    for metric in ('mAP50','mAP50_95','precision','recall'):
        values = [f['yolo_detector'][metric] for f in folds]
        detector[metric] = {'mean':statistics.mean(values),'std':statistics.stdev(values),'values':values}
    methods = {}
    for name in METHODS.values():
        recorded = next(r for r in rows if r['model']==name)
        scores = [f['crop_classifiers'][name] for f in folds]
        methods[name] = {'folds':len(folds), 'cv_macro_f1':float(recorded['mean']), 'cv_std':float(recorded['sample_std']),
                         'fold_metrics':scores, 'fold_values':[s['macro_f1'] for s in scores]}
    return {'scope':'Fold 1 demo weights. Ten-fold detector and ground-truth crop results; no ten-fold end-to-end validation or final locked-test result.',
            'group_basis':config()['group_basis'], 'classes':names(), 'detector':detector, 'methods':methods,
            'fold_counts':[{'fold':f['fold'],'train_images':f['n_train_images'],'validation_images':f['n_validation_images']} for f in folds]}


### classifiers.py


In [ ]:
%%writefile /content/lavee_project/dashboard/classifiers.py
from pathlib import Path
import threading
import math
import json
import cv2
import torch
from torch import nn
from PIL import Image, ImageOps
from torchvision import transforms
import timm
import yolo12_backend
from recurrent_models import RecurrentClassifier

ROOT=Path(__file__).resolve().parent/'research/classification_results'
FILES={'cnn':'CNN_final.pt','transformer':'Transformer_final.pt','cgan_transformer':'cGAN_Transformer_final.pt','lstm':'LSTM_final.pt','bilstm_attention':'BiLSTM_Attention_final.pt'}
LABELS={'yolo':'YOLO only','cnn':'YOLO + CNN','transformer':'YOLO + Transformer','cgan_transformer':'YOLO + cGAN-trained Transformer','lstm':'YOLO + LSTM (spatial patches)','bilstm_attention':'YOLO + BiLSTM + Attention (spatial patches)'}
TRANSFORM=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
CACHE={}
LOCK=threading.Lock()

def validate(technique):
    if technique.startswith('yolo12_'):
        if not yolo12_backend.ready():raise ValueError('YOLO12 package has not passed validation.')
        if technique[7:] not in LABELS:raise ValueError('Unknown classification technique.')
        return
    if technique not in LABELS:raise ValueError('Unknown classification technique.')
    if technique!='yolo' and not (ROOT/FILES[technique]).is_file():raise ValueError('Six-class trained weights are not installed for this technique.')

def load(technique):
    if technique not in CACHE:
        checkpoint=torch.load(ROOT/FILES[technique],map_location='cpu',weights_only=True)
        extractor=timm.create_model(checkpoint['architecture'],pretrained=False,num_classes=0)
        extractor.load_state_dict(checkpoint['extractor'],strict=True)
        if technique in ('lstm','bilstm_attention'):
            if checkpoint.get('input_basis')!='spatial_tokens' or checkpoint.get('grid_size')!=4 or checkpoint.get('input_dim')!=512:
                raise ValueError('Incompatible recurrent checkpoint: expected 16 spatial tokens with 512 features.')
            expected=technique=='bilstm_attention'
            if checkpoint['bidirectional']!=expected or checkpoint['attention']!=expected:raise ValueError('Recurrent architecture mismatch.')
            head=RecurrentClassifier(512,bidirectional=expected,attention=expected)
        else:
            head=nn.Linear(checkpoint['head']['weight'].shape[1],len(checkpoint['classes']))
        if checkpoint['classes']!=['airplane','bird','helicopter','kite','parachute','drone']:raise ValueError('Checkpoint class order does not match dashboard.')
        head.load_state_dict(checkpoint['head'],strict=True)
        CACHE[technique]=(extractor.eval(),head.eval(),checkpoint['classes'])
    return CACHE[technique]

def classify(image,boxes,technique):
    validate(technique)
    if technique.startswith('yolo12_'):
        key=technique[7:]
        return [None]*len(boxes) if key=='yolo' or not boxes else yolo12_backend.classify(image,boxes,key)
    if technique=='yolo' or not boxes:return [None]*len(boxes)
    h,w=image.shape[:2];crops=[]
    for box in boxes:
        x1,y1,x2,y2=box;dx=(x2-x1)*.04;dy=(y2-y1)*.04
        left=max(0,int(x1-dx));top=max(0,int(y1-dy));right=min(w,math.ceil(x2+dx));bottom=min(h,math.ceil(y2+dy))
        if right<=left or bottom<=top:raise ValueError('Invalid detector crop.')
        crop=Image.fromarray(cv2.cvtColor(image[top:bottom,left:right],cv2.COLOR_BGR2RGB))
        crops.append(TRANSFORM(ImageOps.pad(crop,(224,224),color=(114,114,114))))
    with LOCK,torch.inference_mode():
        extractor,head,names=load(technique)
        batch=torch.stack(crops)
        if technique in ('lstm','bilstm_attention'):
            z=nn.functional.adaptive_avg_pool2d(extractor.forward_features(batch),(4,4)).flatten(2).transpose(1,2)
        else:z=extractor(batch)
        probs=head(z).softmax(1)
    return [{'class_id':int(row.argmax()),'class_name':names[int(row.argmax())],'confidence':round(float(row.max()),4)} for row in probs]

def annotate(result,records):
    image=result.plot(labels=False)
    for record in records:
        x,y=map(int,record['box'][:2]);prefix=f"ID {record['track_id']} · " if record.get('track_id') is not None else ''
        label=f"{prefix}{record['class_name']} {record['confidence']:.2f}"
        if record.get('classifier'):label+=f" / YOLO {record['detector_class']}"
        label=label.replace('·','|')
        width=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,.5,1)[0][0]
        cv2.rectangle(image,(max(0,x),max(0,y-24)),(min(image.shape[1],x+width+8),max(24,y)),(35,65,40),-1)
        cv2.putText(image,label,(max(0,x+3),max(17,y-7)),cv2.FONT_HERSHEY_SIMPLEX,.5,(255,255,255),1,cv2.LINE_AA)
    return image

def research_summary():
    report=json.loads((ROOT/'metrics.json').read_text())
    return {'scope':report['scope'],'classes':report['classes'],'methods':{name:{'folds':len(data['folds']),'cv_macro_f1':data['mean_macro_f1'],'cv_std':data['std_macro_f1'],'test_accuracy':data['test']['accuracy'],'test_macro_f1':data['test']['macro_f1']} for name,data in report['methods'].items()}}


### app.py


In [ ]:
%%writefile /content/lavee_project/dashboard/app.py
from pathlib import Path
import base64
import json
import threading
import time
import uuid
import subprocess
import imageio_ffmpeg
import sys
import asyncio
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
from fastapi import FastAPI, File, Form, HTTPException, UploadFile, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from ultralytics import YOLO
import classifiers
import yolo12_backend
import torch
torch.set_num_threads(4)

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / 'vendor'))
def load_model(family='yolov8s'):
    detector = YOLO(str(yolo12_backend.ROOT/'yolo12_best.pt' if family=='yolo12' else ROOT/'models/best.pt'))
    detector.model.names = {key: 'drone' if name.lower() == 'uav' else name
                            for key, name in detector.names.items()}
    if family=='yolo12' and list(detector.names.values()) != yolo12_backend.names():
        raise ValueError('YOLO and classifier class order mismatch.')
    return detector

model = load_model()
models = {'yolov8s':model}

def selected_model(technique):
    family = 'yolo12' if technique.startswith('yolo12_') else 'yolov8s'
    if family not in models:models[family]=load_model(family)
    return models[family]

def inference_size(technique):
    return 416 if technique.startswith('yolo12_') else 640
lock = threading.Lock()
app = FastAPI(title='LAVEE · Flying Object Detection')
app.mount('/static', StaticFiles(directory=ROOT / 'static'), name='static')
app.mount('/demos', StaticFiles(directory=ROOT / 'demos'), name='demos')
(ROOT / 'results').mkdir(exist_ok=True)
app.mount('/results', StaticFiles(directory=ROOT / 'results'), name='results')
jobs = {}
pool = ThreadPoolExecutor(max_workers=1)

@app.get('/')
def home():
    return FileResponse(ROOT / 'static/index.html')

@app.get('/api/model')
def metadata():
    manifest = ROOT / 'demos/manifest.json'
    active = yolo12_backend.ready()
    return {'model': 'YOLOv8s', 'classes': model.names,
            'families': {'yolov8s':{'name':'YOLOv8s','available':True,'package':'Original six-class detector + balanced-crop classifiers','inference_size':640,'research':classifiers.research_summary()},
                         'yolo12':{'name':'YOLO12n','available':bool(active),'package':'Fold 1 demo export','inference_size':416,'research':yolo12_backend.research_summary() if active else None}},
            'techniques':classifiers.LABELS, 'technique_available': {key:key=='yolo' or (classifiers.ROOT/classifiers.FILES[key]).is_file() for key in classifiers.LABELS}, 'research': classifiers.research_summary(),
            'demos': json.loads(manifest.read_text()) if manifest.exists() else []}

def detect(image, confidence, iou, technique="yolo"):
    try: classifiers.validate(technique)
    except ValueError as exc: raise HTTPException(400,str(exc))
    if not 0.05 <= confidence <= 0.95 or not 0.1 <= iou <= 0.9:
        raise HTTPException(400, 'Threshold is outside the supported range.')
    started = time.perf_counter()
    with lock:
        detector=selected_model(technique)
        result = detector.predict(image, conf=confidence, iou=iou, imgsz=inference_size(technique), verbose=False)[0]
    detections = [{'class_id': int(b.cls.item()), 'class_name': detector.names[int(b.cls.item())],
                   'confidence': round(float(b.conf.item()), 4),
                   'box': [round(float(v), 1) for v in b.xyxy[0].tolist()]} for b in result.boxes]
    predictions=classifiers.classify(image,[d['box'] for d in detections],technique)
    for record,pred in zip(detections,predictions):
        record.update(detector_class=record['class_name'],detector_confidence=record['confidence'],classifier=technique if pred else None)
        if pred: record.update(pred)
    ok, encoded = cv2.imencode('.jpg', classifiers.annotate(result,detections), [cv2.IMWRITE_JPEG_QUALITY, 92])
    if not ok:
        raise HTTPException(500, 'Could not render the detection result.')
    return {'image': 'data:image/jpeg;base64,' + base64.b64encode(encoded).decode(),
            'detections': detections, 'elapsed_ms': round((time.perf_counter()-started)*1000),
            'width': image.shape[1], 'height': image.shape[0], 'technique': technique}

@app.post('/api/detect')
async def upload(file: UploadFile = File(...), confidence: float = Form(0.25), iou: float = Form(0.45), technique: str = Form("yolo")):
    content = await file.read(20 * 1024 * 1024 + 1)
    if len(content) > 20 * 1024 * 1024:
        raise HTTPException(413, 'Please upload an image smaller than 20 MB.')
    image = cv2.imdecode(np.frombuffer(content, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise HTTPException(400, 'Please select a valid JPG, PNG, or WebP image.')
    if image.shape[0] * image.shape[1] > 40_000_000:
        raise HTTPException(400, 'Please resize this image to under 40 megapixels.')
    return detect(image, confidence, iou, technique)

@app.post('/api/demo/{class_id}')
def demo(class_id: int, confidence: float = Form(0.25), iou: float = Form(0.45), technique: str = Form("yolo")):
    if class_id not in model.names:
        raise HTTPException(404, 'Unknown class.')
    path = ROOT / 'demos' / f'{class_id}.jpg'
    image = cv2.imread(str(path)) if path.exists() else None
    if image is None:
        raise HTTPException(404, 'Demo sample is unavailable.')
    return detect(image, confidence, iou, technique)

def process_video(job_id, source, confidence, iou, remove_source=False, technique="yolo"):
    cap = cv2.VideoCapture(str(source))
    writer = None
    raw = ROOT / 'results' / f'{job_id}_raw.mp4'
    output = ROOT / 'results' / f'{job_id}.mp4'
    started = time.perf_counter()
    try:
        classifiers.validate(technique)
        video_model = load_model('yolo12' if technique.startswith('yolo12_') else 'yolov8s')
        if not cap.isOpened():
            raise ValueError('Cannot read this video. Please use an MP4 file.')
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        if not 1 <= fps <= 240:
            fps = 25
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total / fps > 120:
            raise ValueError('Please upload a video up to 2 minutes long.')
        frames = 0
        counts = {}
        peaks = {}
        trails = defaultdict(lambda: deque(maxlen=45))
        seen_tracks = set()
        while True:
            while jobs[job_id].get('paused') and not jobs[job_id].get('stop'):
                time.sleep(0.05)
            if jobs[job_id].get('stop'):
                break
            frame_started = time.perf_counter()
            ok, frame = cap.read()
            if not ok:
                break
            if frames >= int(fps * 120):
                break
            h, w = frame.shape[:2]
            if max(h, w) > 960:
                frame = cv2.resize(frame, (int(w*960/max(h,w)), int(h*960/max(h,w))))
            frame = frame[:frame.shape[0]//2*2, :frame.shape[1]//2*2]
            with lock:
                result = video_model.track(frame, persist=True, tracker='bytetrack.yaml', conf=confidence, iou=iou, imgsz=inference_size(technique), verbose=False)[0]
            present = set()
            live_tracks = []
            for box in result.boxes:
                name = video_model.names[int(box.cls.item())]
                present.add(name)
                peaks[name] = max(peaks.get(name, 0), float(box.conf.item()))
                track_id = int(box.id.item()) if box.id is not None else None
                coords = [round(float(v), 1) for v in box.xyxy[0].tolist()]
                live_tracks.append({'track_id': track_id, 'class_name': name, 'confidence': round(float(box.conf.item()),4), 'box': coords})
                if track_id is not None:
                    seen_tracks.add(track_id)
                    trails[track_id].append((int((coords[0]+coords[2])/2), int((coords[1]+coords[3])/2)))
            predictions=classifiers.classify(frame,[t['box'] for t in live_tracks],technique)
            present=set()
            for record,pred in zip(live_tracks,predictions):
                record.update(detector_class=record['class_name'],detector_confidence=record['confidence'],classifier=technique if pred else None)
                if pred: record.update(pred)
                present.add(record['class_name'])
                peaks[record['class_name']]=max(peaks.get(record['class_name'],0),record['confidence'])
            for name in present:
                counts[name] = counts.get(name, 0) + 1
            if writer is None:
                writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame.shape[1], frame.shape[0]))
                if not writer.isOpened():
                    raise ValueError('Could not create the output video.')
            annotated = classifiers.annotate(result,live_tracks)
            for track in live_tracks:
                points = trails.get(track['track_id'], [])
                if len(points) > 1:
                    cv2.polylines(annotated, [np.array(points, np.int32)], False, (100,230,180), 2)
            writer.write(annotated)
            frames += 1
            encoded_ok, jpeg = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 80])
            if encoded_ok:
                jobs[job_id]['live'] = {'type': 'frame', 'technique': technique, 'frame': frames, 'image': 'data:image/jpeg;base64,'+base64.b64encode(jpeg).decode(), 'tracks': live_tracks, 'unique_tracks': len(seen_tracks), 'fps': round(1/max(time.perf_counter()-frame_started,0.001),1), 'video_seconds': round(frames/fps,2), 'progress': min(99,round(frames/max(total,1)*100))}
            jobs[job_id].update(progress=min(95, round(frames/max(total,1)*95)), frames=frames)
            time.sleep(max(0, 1/fps-(time.perf_counter()-frame_started)))
        if writer:
            writer.release()
            writer = None
        if not frames:
            raise ValueError('The video contains no readable frames.')
        subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), '-y', '-loglevel', 'error', '-i', str(raw), '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(output)], check=True, capture_output=True)
        payload = {'status': 'complete', 'progress': 100, 'video': f'/results/{job_id}.mp4', 'frames': frames,
                   'technique': technique, 'stopped': bool(jobs[job_id].get('stop')), 'unique_tracks': len(seen_tracks),
                   'elapsed_ms': round((time.perf_counter()-started)*1000),
                   'detections': [{'class_name': name, 'frames_detected': count, 'confidence': round(peaks[name],4)} for name,count in counts.items()]}
        jobs[job_id].update(payload)
        (ROOT / 'results' / f'{job_id}.json').write_text(json.dumps(payload, indent=2))
    except Exception as exc:
        jobs[job_id].update(status='failed', error=str(exc))
    finally:
        cap.release()
        if writer:
            writer.release()
        raw.unlink(missing_ok=True)
        if remove_source:
            source.unlink(missing_ok=True)

@app.post('/api/video')
async def video_upload(file: UploadFile = File(...), confidence: float = Form(0.25), iou: float = Form(0.45), technique: str = Form("yolo")):
    if not 0.05 <= confidence <= 0.95 or not 0.1 <= iou <= 0.9:
        raise HTTPException(400, 'Invalid threshold.')
    try: classifiers.validate(technique)
    except ValueError as exc: raise HTTPException(400,str(exc))
    job_id = uuid.uuid4().hex
    source = ROOT / 'results' / f'{job_id}_upload.mp4'
    size = 0
    with source.open('wb') as target:
        while content := await file.read(1024*1024):
            size += len(content)
            if size > 100*1024*1024:
                target.close()
                source.unlink(missing_ok=True)
                raise HTTPException(413, 'Please upload a video smaller than 100 MB.')
            target.write(content)
    jobs[job_id] = {'status': 'processing', 'progress': 0}
    pool.submit(process_video, job_id, source, confidence, iou, True, technique=technique)
    return {'job_id': job_id}

@app.post('/api/video-demo/{class_id}')
def video_demo(class_id: int, confidence: float = Form(0.25), iou: float = Form(0.45), technique: str = Form("yolo")):
    source = ROOT / 'demos' / f'{class_id}.mp4'
    if class_id not in model.names or not source.exists():
        raise HTTPException(404, 'Video demo unavailable.')
    if not 0.05 <= confidence <= 0.95 or not 0.1 <= iou <= 0.9:
        raise HTTPException(400, 'Invalid threshold.')
    try: classifiers.validate(technique)
    except ValueError as exc: raise HTTPException(400,str(exc))
    job_id = uuid.uuid4().hex
    jobs[job_id] = {'status': 'processing', 'progress': 0}
    pool.submit(process_video, job_id, source, confidence, iou, technique=technique)
    return {'job_id': job_id}

@app.get('/api/jobs/{job_id}')
def job_status(job_id: str):
    if job_id not in jobs:
        raise HTTPException(404, 'Job not found.')
    return {k:v for k,v in jobs[job_id].items() if k != 'live'}

@app.post('/api/library/{clip_id}')
def library_demo(clip_id: str, confidence: float = Form(0.25), iou: float = Form(0.45), technique: str = Form("yolo")):
    manifest = json.loads((ROOT / 'demos/manifest.json').read_text())
    clip = next((clip for item in manifest for clip in item.get('clips',[]) if clip['id'] == clip_id), None)
    if clip is None:
        raise HTTPException(404, 'Demo clip not found.')
    source = (ROOT / 'demos' / clip['file']).resolve()
    if source.parent != (ROOT / 'demos').resolve() or not source.is_file():
        raise HTTPException(404, 'Demo file unavailable.')
    if not 0.05 <= confidence <= 0.95 or not 0.1 <= iou <= 0.9:
        raise HTTPException(400, 'Invalid threshold.')
    try: classifiers.validate(technique)
    except ValueError as exc: raise HTTPException(400,str(exc))
    job_id = uuid.uuid4().hex
    jobs[job_id] = {'status': 'processing', 'progress': 0}
    pool.submit(process_video, job_id, source, confidence, iou, technique=technique)
    return {'job_id': job_id}

@app.post('/api/jobs/{job_id}/{action}')
def control_job(job_id: str, action: str):
    if job_id not in jobs:
        raise HTTPException(404, 'Job not found.')
    if action not in ('pause', 'resume', 'stop'):
        raise HTTPException(400, 'Unknown action.')
    if action == 'stop':
        jobs[job_id]['stop'] = True
    else:
        jobs[job_id]['paused'] = action == 'pause'
    return {'ok': True}

@app.websocket('/api/live/{job_id}')
async def live_stream(socket: WebSocket, job_id: str):
    await socket.accept()
    if job_id not in jobs:
        await socket.send_json({'type':'error', 'error':'Job not found.'})
        await socket.close()
        return
    last_frame = 0
    try:
        while True:
            job = jobs[job_id]
            packet = job.get('live')
            if packet and packet['frame'] != last_frame:
                await socket.send_json(packet)
                last_frame = packet['frame']
            if job['status'] in ('complete','failed'):
                await socket.send_json({'type':'end', **{k:v for k,v in job.items() if k != 'live'}})
                break
            await asyncio.sleep(0.03)
    except WebSocketDisconnect:
        pass


@app.websocket('/api/webcam')
async def webcam(socket: WebSocket):
    await socket.accept()
    try:
        settings=await socket.receive_json()
        technique=settings.get('technique','yolo')
        classifiers.validate(technique)
        confidence=float(settings.get('confidence',.25));iou=float(settings.get('iou',.45))
        if not .05<=confidence<=.95 or not .1<=iou<=.9:raise ValueError('Invalid threshold.')
        detector=await asyncio.to_thread(load_model,'yolo12' if technique.startswith('yolo12_') else 'yolov8s')
        frame_id=0;seen=set()
        def infer_frame(image):
            started=time.perf_counter()
            with lock:
                result=detector.track(image,persist=True,tracker='bytetrack.yaml',conf=confidence,iou=iou,imgsz=inference_size(technique),verbose=False)[0]
            records=[]
            for box in result.boxes:
                records.append({'track_id':int(box.id.item()) if box.id is not None else None,
                    'class_name':detector.names[int(box.cls.item())],'confidence':round(float(box.conf.item()),4),
                    'box':[round(float(v),1) for v in box.xyxy[0].tolist()]})
            for record,pred in zip(records,classifiers.classify(image,[r['box'] for r in records],technique)):
                record.update(detector_class=record['class_name'],detector_confidence=record['confidence'],classifier=technique if pred else None)
                if pred:record.update(pred)
                if record['track_id'] is not None:seen.add(record['track_id'])
            ok,jpg=cv2.imencode('.jpg',classifiers.annotate(result,records),[cv2.IMWRITE_JPEG_QUALITY,80])
            if not ok:raise ValueError('Could not render frame.')
            return {'type':'frame','technique':technique,'frame':frame_id,'tracks':records,'unique_tracks':len(seen),
                    'image':'data:image/jpeg;base64,'+base64.b64encode(jpg).decode(),'fps':round(1/max(.001,time.perf_counter()-started),1),
                    'video_seconds':0,'progress':0}
        while True:
            content=await socket.receive_bytes()
            if len(content)>2*1024*1024:raise ValueError('Camera frame too large.')
            image=cv2.imdecode(np.frombuffer(content,np.uint8),cv2.IMREAD_COLOR)
            if image is None or max(image.shape[:2])>1280:raise ValueError('Invalid camera frame.')
            frame_id+=1
            await socket.send_json(await asyncio.to_thread(infer_frame,image))
    except WebSocketDisconnect:
        pass
    except Exception as exc:
        await socket.send_json({'type':'error','error':str(exc)})
        await socket.close()


### static/index.html


In [ ]:
%%writefile /content/lavee_project/dashboard/static/index.html
<!doctype html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>LAVEE · Detection Studio</title>
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@400;500;600;700&display=swap');
:root{--bg:#f4f5f0;--ink:#17271e;--muted:#6d796f;--line:#dce1d8;--green:#244f36;--lime:#d6ee91}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:'DM Sans',Arial,sans-serif}button,input{font:inherit}button{cursor:pointer}button:disabled{opacity:.5;cursor:wait}header{height:84px;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;padding:0 4vw;background:#fafbf7}.brand{font:700 26px 'Space Grotesk',sans-serif;letter-spacing:3px;display:flex;align-items:center;gap:12px}.brand span{background:var(--green);color:var(--lime);border-radius:10px;padding:7px 10px;font-size:22px;letter-spacing:0}.pill{font-size:12px;border:1px solid var(--line);padding:8px 13px;border-radius:30px;display:inline-flex;align-items:center;gap:8px}.dot{width:7px;height:7px;background:#6c9839;border-radius:50%}main{max-width:1480px;margin:auto;padding:42px 4vw}.eyebrow{font-size:11px;letter-spacing:2px;font-weight:700;color:var(--muted)}h1{font:500 clamp(32px,4vw,48px) 'Space Grotesk',sans-serif;margin:10px 0 12px;letter-spacing:-2px}.intro{color:var(--muted);font-size:14px;margin:0}.heading{display:flex;align-items:center;justify-content:space-between;margin-bottom:30px}.badge{background:var(--lime);padding:12px 17px;border-radius:12px;font-size:12px}.grid{display:grid;grid-template-columns:300px 1fr;gap:22px}.panel{background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}.settings{padding:24px;height:fit-content}h2{font:600 17px 'Space Grotesk',sans-serif;margin:0 0 22px}.label{font-size:12px;font-weight:700;display:flex;justify-content:space-between;margin:24px 0 12px}input[type=range]{width:100%;accent-color:var(--green)}.hint{color:var(--muted);font-size:11px;line-height:1.7}.divider{height:1px;background:var(--line);margin:24px 0}.classes{display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:12px}.class{display:flex;align-items:center;gap:7px}.class i{width:7px;height:7px;border-radius:2px;background:var(--green)}.model{background:#f4f6ef;padding:14px;border-radius:10px;line-height:1.8;font-size:12px}.model small{color:var(--muted)}.workspace{min-width:0}.toolbar{padding:16px 22px;display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--line);gap:12px}.tabs{background:#f0f2ec;padding:4px;border-radius:10px;display:flex;gap:4px}.tab{border:0;background:none;padding:9px 18px;border-radius:7px;font-size:12px;color:var(--muted)}.tab.active{background:white;color:var(--ink);box-shadow:0 1px 3px #0001}.status{font-size:11px;color:var(--muted)}.input-area{padding:24px}.drop{border:1.5px dashed #bdcbbb;background:#fafbf7;border-radius:14px;min-height:210px;display:flex;align-items:center;justify-content:center;flex-direction:column;gap:10px;text-align:center;cursor:pointer}.drop.over{background:#e8f0dc;border-color:var(--green)}.upload-icon{border:1px solid var(--line);background:white;border-radius:14px;padding:12px 18px;font-size:27px}.drop strong{font-size:14px}.drop span{font-size:12px;color:var(--muted)}.primary{background:var(--green);border:0;color:white;padding:12px 22px;border-radius:9px;font-weight:600;font-size:12px}.actions{display:flex;justify-content:space-between;align-items:center;margin-top:16px;gap:12px}.filename{font-size:12px;color:var(--muted);overflow:hidden;text-overflow:ellipsis}.demo-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}.demo{padding:0;border:1px solid var(--line);border-radius:11px;overflow:hidden;background:#fff;text-align:left}.demo img{width:100%;height:110px;object-fit:cover;background:#e8ede5}.demo div{padding:11px;display:flex;justify-content:space-between;font-size:12px}.demo:hover,.demo.selected{border-color:var(--green);box-shadow:0 0 0 1px var(--green)}.result-head{padding:0 24px 16px;display:flex;align-items:center;justify-content:space-between;font-size:12px}.result-head strong{font-size:14px}.canvas{margin:0 24px;background:#edf0e9;border-radius:12px;min-height:270px;display:flex;align-items:center;justify-content:center;overflow:hidden;position:relative}.empty{text-align:center;color:#82907f;font-size:12px;line-height:2}.empty b{font-size:42px;font-weight:400}.canvas img{display:block;max-width:100%;max-height:650px;object-fit:contain}.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:1px;background:var(--line);margin:20px 24px;border:1px solid var(--line);border-radius:10px;overflow:hidden}.metric{background:white;padding:15px}.metric small{display:block;color:var(--muted);font-size:10px;text-transform:uppercase;letter-spacing:1px}.metric strong{font:500 23px 'Space Grotesk',sans-serif;display:block;margin-top:6px}.table-wrap{padding:0 24px 24px;overflow:auto}table{width:100%;border-collapse:collapse;font-size:12px}th{text-align:left;font-size:10px;color:var(--muted);font-weight:500;padding:12px 8px;border-bottom:1px solid var(--line)}td{padding:12px 8px;border-bottom:1px solid #edf0e8}.confidence{background:#eef4e4;color:var(--green);padding:5px 9px;border-radius:5px}.secondary{border:1px solid var(--line);background:white;padding:7px 11px;border-radius:7px;color:var(--ink);font-size:11px;text-decoration:none}.downloads{display:flex;gap:7px}footer{font-size:11px;color:var(--muted);padding:24px 0;display:flex;justify-content:space-between}.error{background:#fff1ee;color:#9a3827;border:1px solid #eec8be;padding:13px;margin:0 24px 18px;border-radius:8px;font-size:12px}[hidden]{display:none!important}@media(max-width:850px){.grid{grid-template-columns:1fr}.settings{display:grid;grid-template-columns:1fr 1fr;gap:0 25px}.settings h2{grid-column:1/-1}.settings .divider,.settings .model,.settings .classes-title,.settings .classes{display:none}.settings .label{margin-top:0}.heading .badge{display:none}main{padding-top:28px}.toolbar{flex-wrap:wrap}.demo-grid{grid-template-columns:repeat(2,1fr)}footer{gap:16px}.result-head{flex-wrap:wrap;gap:12px}}
</style></head><body>
<header><div class="brand"><span>⌁</span>LAVEE</div><div class="pill"><i class="dot"></i>Flying Object Research</div></header>
<main><div class="heading"><div><div class="eyebrow">DETECTION STUDIO / 01</div><h1>A clearer view of the sky.</h1><p class="intro">Upload an image or video, or explore six classes with live model detection.</p></div><div class="badge">6 classes · 2 detectors · 5 hybrid techniques</div></div>
<div class="grid"><aside class="panel settings"><h2>Detection settings</h2><div style="grid-column:1/-1"><label class="label" for="family">Detection model</label><select id="family" style="width:100%;padding:11px;border:1px solid #dce1d8;border-radius:8px;background:#fff"><option value="yolov8s">YOLOv8s</option><option value="yolo12">YOLO12n</option></select><p class="hint" id="package-info"></p><label class="label" for="technique">Classification technique</label><select id="technique" style="width:100%;padding:11px;border:1px solid #dce1d8;border-radius:8px;background:#fff"><option value="yolo">YOLO only</option><option value="cnn">YOLO + CNN</option><option value="transformer">YOLO + Transformer</option><option value="cgan_transformer">YOLO + cGAN-trained Transformer</option><option value="lstm" disabled>YOLO + LSTM (spatial patches) — training required</option><option value="bilstm_attention" disabled>YOLO + BiLSTM + Attention (spatial patches) — training required</option></select><p class="hint">YOLO finds and tracks objects. The selected classifier labels each detected crop. First use loads the model.</p><p class="hint">LSTM and BiLSTM use spatial image patches, not consecutive video frames.</p><details><summary style="font-size:12px;cursor:pointer">Recorded training results</summary><div id="research-results" class="hint"></div><p class="hint">Scores are for ground-truth crops, not full-video detection. cGAN was used during training; it does not modify live footage.</p></details></div><div><label class="label" for="conf">Confidence threshold <span id="conf-value">25%</span></label><input id="conf" type="range" min="5" max="95" value="25"><p class="hint">Lower values show more candidates. Higher values retain more confident detections.</p></div><div><label class="label" for="iou">Overlap threshold (IoU) <span id="iou-value">45%</span></label><input id="iou" type="range" min="10" max="90" value="45"><p class="hint">Controls removal of overlapping boxes. Apply changes by running detection again.</p></div><div class="divider"></div><div class="label classes-title">SUPPORTED CLASSES</div><div id="classes" class="classes"></div><div class="divider"></div><div class="model"><strong id="model-name">YOLOv8s</strong><br><small id="model-info">Your fine-tuned checkpoint</small></div></aside>
<section class="panel workspace"><div class="toolbar"><div class="tabs"><button class="tab active" data-mode="upload">Upload media</button><button class="tab" data-mode="demo">Explore demos</button></div><span id="status" class="status" role="status" aria-live="polite">Loading model information…</span></div>
<div class="input-area"><div id="upload-pane"><input type="file" id="file" accept="image/jpeg,image/png,image/webp,video/mp4,video/quicktime,video/x-msvideo" hidden><div class="drop" id="drop" role="button" tabindex="0" aria-label="Choose an image or video"><div class="upload-icon">↥</div><strong>Drop your image or video here</strong><span>JPG, PNG, WebP · 20 MB / Video · 100 MB, up to 2 minutes</span></div><div class="actions"><span id="filename" class="filename">No image selected</span><button id="camera-start" class="secondary">Use webcam</button><button id="camera-stop" class="secondary" hidden>Stop camera</button><button id="run" class="primary" disabled>Run detection ↗</button></div></div><div id="demo-pane" hidden><p class="hint" style="margin-top:0">Choose a class and video, or try Mixed classes for simultaneous detection across four camera views. Kite and parachute currently use clearly labeled still-image clips. These curated demos are not an accuracy benchmark.</p><div id="demos" class="demo-grid"></div></div></div>
<div id="error" class="error" role="alert" hidden></div><div class="result-head"><strong>Live detection & tracking</strong><div id="live-controls" hidden><button id="pause-live" class="secondary">Pause</button> <button id="stop-live" class="secondary">Stop</button></div><div class="downloads" id="downloads" hidden><a id="save-image" class="secondary" download="lavee-detections.jpg">Save image ↓</a><button id="save-json" class="secondary">Export results ↓</button></div></div><div class="canvas"><div class="empty" id="empty"><b>⌖</b><br>Your detections will appear here<br>Bounding boxes · Class labels · Confidence</div><video id="video-result" controls playsinline hidden style="width:100%;max-height:650px"></video><img id="result" alt="Image with model-predicted bounding boxes" hidden></div>
<div class="metrics"><div class="metric"><small>Active objects</small><strong id="count">—</strong></div><div class="metric"><small>Tracks seen</small><strong id="found">—</strong></div><div class="metric"><small>Inference speed</small><strong id="time">—</strong></div></div><div class="table-wrap"><table><thead><tr><th>TRACK / OBJECT</th><th>CLASS / YOLO LABEL</th><th>CONFIDENCE</th><th>BOX · PX / VIDEO FRAMES</th></tr></thead><tbody id="rows"><tr><td colspan="4" style="color:var(--muted)">Run detection to inspect the results.</td></tr></tbody></table></div></section></div>
<footer><span>LAVEE / Flying Object Detection</span><span>Predictions may include missed objects or incorrect labels.</span></footer></main>
<script>
const $=id=>document.getElementById(id);let chosen=null,last=null,busy=false;
for(const id of ['conf','iou'])$(id).addEventListener('input',()=>$(id+'-value').textContent=$(id).value+'%');
function clearResult(){last=null;$('video-result').pause();$('video-result').hidden=true;$('video-result').removeAttribute('src');$('save-image').download='lavee-detections.jpg';$('save-image').textContent='Save image ↓';$('result').hidden=true;$('result').removeAttribute('src');$('empty').hidden=false;$('downloads').hidden=true;for(const id of ['count','found','time'])$(id).textContent='—';$('rows').replaceChildren();$('error').hidden=true;}
document.querySelectorAll('[data-mode]').forEach(b=>b.onclick=()=>{if(busy)return;document.querySelectorAll('[data-mode]').forEach(t=>t.classList.toggle('active',t===b));$('upload-pane').hidden=b.dataset.mode!=='upload';$('demo-pane').hidden=b.dataset.mode!=='demo';clearResult();});
function select(file){if(!file)return;clearResult();if(file.size>(file.type.startsWith('video/')?100:20)*1024*1024){showError('File too large: images up to 20 MB; videos up to 100 MB.');chosen=null;$('run').disabled=true;return;}chosen=file;$('filename').textContent=file.name;$('run').disabled=false;}
$('drop').onclick=()=>$('file').click();$('drop').onkeydown=e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();$('file').click();}};$('file').onchange=e=>select(e.target.files[0]);
for(const event of ['dragenter','dragover'])$('drop').addEventListener(event,e=>{e.preventDefault();$('drop').classList.add('over');});for(const event of ['dragleave','drop'])$('drop').addEventListener(event,e=>{e.preventDefault();$('drop').classList.remove('over');});$('drop').addEventListener('drop',e=>{if(!busy)select(e.dataTransfer.files[0]);});
function showError(message){$('error').hidden=false;$('error').textContent=message;}
async function inferImage(url,file){if(busy)return;clearResult();busy=true;$('technique').disabled=true;$('family').disabled=true;document.querySelectorAll('button').forEach(b=>b.disabled=true);$('file').disabled=true;$('status').textContent='Detecting objects…';let form=new FormData();form.append('technique',selectedTechnique());form.append('confidence',$('conf').value/100);form.append('iou',$('iou').value/100);if(file)form.append('file',file);
try{let response=await fetch(url,{method:'POST',body:form});let data=await response.json();if(!response.ok)throw new Error(typeof data.detail==='string'?data.detail:'Detection could not complete.');last={...data};delete last.image;$('result').src=data.image;$('result').hidden=false;$('empty').hidden=true;$('count').textContent=data.detections.length;$('found').textContent=new Set(data.detections.map(d=>d.class_id)).size;$('time').textContent=(data.elapsed_ms/1000).toFixed(2)+' s';$('save-image').href=data.image;$('downloads').hidden=false;$('rows').replaceChildren();data.detections.forEach((d,i)=>{let row=document.createElement('tr');for(const value of [String(i+1).padStart(2,'0'),(d.classifier?d.class_name+' (YOLO: '+d.detector_class+')':d.class_name),(d.confidence*100).toFixed(1)+'%',d.box.join(', ')]){let cell=document.createElement('td');cell.textContent=value;row.appendChild(cell);}$('rows').appendChild(row);});if(!data.detections.length){let row=$('rows').insertRow();let cell=row.insertCell();cell.colSpan=4;cell.textContent='No objects detected at this threshold. Try a lower confidence or another image.';}$('status').textContent='Detection complete';}catch(e){showError(e.message);$('status').textContent='Detection failed';}finally{busy=false;$('technique').disabled=false;$('family').disabled=false;document.querySelectorAll('button').forEach(b=>b.disabled=false);$('file').disabled=false;$('run').disabled=!chosen;}}
$('run').onclick=()=>chosen&&infer('/api/detect',chosen);$('save-json').onclick=()=>{if(!last)return;let url=URL.createObjectURL(new Blob([JSON.stringify(last,null,2)],{type:'application/json'}));let a=document.createElement('a');a.href=url;a.download='lavee-results.json';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};

let modelMetadata=null;
function selectedTechnique(){return ($('family').value==='yolo12'?'yolo12_':'')+$('technique').value;}
function renderFamily(){
 if(!modelMetadata)return;
 const family=$('family').value, f=modelMetadata.families[family], r=f.research;
 $('model-name').textContent=f.name;$('model-info').textContent='Inference size · '+f.inference_size+' px · Local processing';
 $('package-info').textContent=f.package;
 for(const option of $('technique').options){option.textContent=modelMetadata.techniques[option.value].replace('YOLO',f.name);}
 const root=$('research-results');root.replaceChildren();
 const add=text=>{let p=document.createElement('p');p.textContent=text;root.appendChild(p);};
 if(r.detector){for(const [key,label] of [['mAP50','Detection mAP50'],['mAP50_95','Detection mAP50–95']]){let v=r.detector[key];add(label+': '+(v.mean*100).toFixed(2)+'% ± '+(v.std*100).toFixed(2)+' (10 folds)');}}
 else add('Detector: existing YOLOv8s checkpoint; detector 10-fold results were not supplied.');
 for(const [name,v] of Object.entries(r.methods)){
  add(name+': crop CV macro-F1 '+(v.cv_macro_f1*100).toFixed(2)+'% ± '+(v.cv_std*100).toFixed(2)+' · '+v.folds+' folds'+(v.test_accuracy==null?'':' · Test accuracy '+(v.test_accuracy*100).toFixed(1)+'%'));
 }
 if(r.detector){
  let label=document.createElement('label');label.textContent='Fold performance';root.appendChild(label);
  let select=document.createElement('select');select.style.cssText='width:100%;padding:7px;margin:8px 0';
  for(const [key,name] of [['detector','Detector mAP50–95'],...Object.keys(r.methods).map(k=>[k,k+' crop F1'])]){let o=document.createElement('option');o.value=key;o.textContent=name;select.appendChild(o);}root.appendChild(select);
  let chart=document.createElement('div');root.appendChild(chart);
  function draw(){const v=select.value==='detector'?r.detector.mAP50_95.values:r.methods[select.value].fold_values;chart.replaceChildren();
   const ns='http://www.w3.org/2000/svg';let svg=document.createElementNS(ns,'svg');svg.setAttribute('viewBox','0 0 280 120');svg.setAttribute('role','img');svg.setAttribute('aria-label',select.selectedOptions[0].text+' across folds 1 to 10, vertical scale 0 to 100 percent');
   v.forEach((score,i)=>{let bar=document.createElementNS(ns,'rect');bar.setAttribute('x',i*27+6);bar.setAttribute('y',100-score*90);bar.setAttribute('width',18);bar.setAttribute('height',score*90);bar.setAttribute('fill','#244f36');let title=document.createElementNS(ns,'title');title.textContent='Fold '+(i+1)+': '+(score*100).toFixed(2)+'%';bar.appendChild(title);svg.appendChild(bar);let t=document.createElementNS(ns,'text');t.setAttribute('x',i*27+8);t.setAttribute('y',116);t.setAttribute('font-size',10);t.textContent=i+1;svg.appendChild(t);});chart.appendChild(svg);}
  select.onchange=draw;draw();
  add('Split basis: '+r.group_basis+'. Source recording groups were unavailable, so related-frame leakage cannot be ruled out.');
 }
 add(r.scope);
}
$('family').onchange=()=>{clearResult();renderFamily();};

fetch('/api/model').then(r=>{if(!r.ok)throw new Error('Model information unavailable');return r.json();}).then(data=>{Object.entries(data.techniques).forEach(([key,name])=>{let option=Array.from($('technique').options).find(o=>o.value===key);if(option){option.disabled=!data.technique_available[key];option.textContent=name+(option.disabled?' — training required':'');}});modelMetadata=data;Object.entries(data.families).forEach(([key,f])=>{document.querySelector('#family option[value='+key+']').disabled=!f.available;});renderFamily();Object.values(data.classes).forEach(name=>{let div=document.createElement('div');div.className='class';let dot=document.createElement('i');div.append(dot,document.createTextNode(name));$('classes').appendChild(div);});data.demos.forEach(d=>{let card=document.createElement('div');card.className='demo';let img=document.createElement('img');img.src=d.image;img.alt=d.name+' demo';let title=document.createElement('div');title.textContent=d.name;title.style.fontWeight='700';let select=document.createElement('select');select.setAttribute('aria-label',d.name+' video');select.style.cssText='margin:0 10px 10px;width:calc(100% - 20px);padding:8px;border:1px solid #dce1d8;border-radius:6px;background:white;font-size:12px';let clips=d.clips||[];for(const clip of clips){let option=document.createElement('option');option.value=clip.id;option.textContent=clip.title;select.appendChild(option);}let note=document.createElement('small');note.style.cssText='display:block;padding:0 11px 12px;color:#6d796f;font-size:10px;min-height:40px';function update(){let clip=clips.find(c=>c.id===select.value);if(clip){img.src=clip.image;note.textContent=clip.kind;}}select.onchange=update;update();let play=document.createElement('button');play.className='primary';play.textContent='Start live tracking ↗';play.style.cssText='margin:0 10px 12px;width:calc(100% - 20px)';play.onclick=()=>{if(busy)return;document.querySelectorAll('.demo').forEach(c=>c.classList.toggle('selected',c===card));infer('/api/library/'+encodeURIComponent(select.value));};card.append(img,title,select,note,play);$('demos').appendChild(card);});$('status').textContent='● Model ready';}).catch(e=>{showError(e.message);$('status').textContent='Service unavailable';});

let currentJob=null,paused=false;
function drawLive(data){
$('empty').hidden=true;$('video-result').hidden=true;$('result').hidden=false;$('result').src=data.image;
$('count').textContent=data.tracks.length+' active';$('found').textContent=data.unique_tracks+' tracks';$('time').textContent=data.fps+' FPS';
$('status').textContent=(paused?'PAUSED · ':'LIVE · ')+$('technique').selectedOptions[0].text+' · Frame '+data.frame+' · '+data.video_seconds+' s · '+data.progress+'%';
$('rows').replaceChildren();for(const d of data.tracks){let row=$('rows').insertRow();for(const v of [d.track_id===null?'Pending':'ID '+d.track_id,(d.classifier?d.class_name+' (YOLO: '+d.detector_class+')':d.class_name),(d.confidence*100).toFixed(1)+'%',d.box.join(', ')])row.insertCell().textContent=v;}
if(!data.tracks.length){let row=$('rows').insertRow();let cell=row.insertCell();cell.colSpan=4;cell.textContent='No active detections in this frame.';}
}
async function infer(url,file){
if(!url.includes('video-demo')&&!url.includes('/library/')&&!(file&&(/\.(mp4|mov|avi|mkv|webm)$/i.test(file.name)||file.type.startsWith('video/'))))return inferImage(url,file);
if(busy)return;clearResult();busy=true;$('technique').disabled=true;$('family').disabled=true;document.querySelectorAll('button').forEach(b=>b.disabled=true);$('file').disabled=true;document.querySelectorAll('#demos select').forEach(x=>x.disabled=true);$('status').textContent='Starting live tracking…';let form=new FormData();form.append('technique',selectedTechnique());form.append('confidence',$('conf').value/100);form.append('iou',$('iou').value/100);if(file)form.append('file',file);
try{let response=await fetch(file?'/api/video':url,{method:'POST',body:form});let task=await response.json();if(!response.ok)throw new Error(typeof task.detail==='string'?task.detail:'Unable to open video.');currentJob=task.job_id;paused=false;$('live-controls').hidden=false;$('pause-live').disabled=false;$('stop-live').disabled=false;$('pause-live').textContent='Pause';
let data=await new Promise((resolve,reject)=>{let ws=new WebSocket((location.protocol==='https:'?'wss://':'ws://')+location.host+'/api/live/'+currentJob);let done=false;ws.onmessage=e=>{let packet=JSON.parse(e.data);if(packet.type==='frame')drawLive(packet);if(packet.type==='error'){done=true;ws.close();reject(new Error(packet.error));}if(packet.type==='end'){done=true;ws.close();packet.status==='failed'?reject(new Error(packet.error)):resolve(packet);}};ws.onerror=()=>reject(new Error('Live connection interrupted. Please retry.'));ws.onclose=()=>{if(!done)reject(new Error('Live stream disconnected. Please retry.'));};});
last=data;$('save-image').href=data.video;$('save-image').download='lavee-tracked.mp4';$('save-image').textContent='Save recording ↓';$('downloads').hidden=false;$('status').textContent=(data.stopped?'Stopped':'Tracking complete')+' · '+data.frames+' frames · '+data.unique_tracks+' tracks';
}catch(e){showError(e.message);$('status').textContent='Live tracking failed';if(currentJob)fetch('/api/jobs/'+currentJob+'/stop',{method:'POST'}).catch(()=>{});}finally{currentJob=null;busy=false;$('technique').disabled=false;$('family').disabled=false;document.querySelectorAll('#demos select').forEach(x=>x.disabled=false);$('live-controls').hidden=true;document.querySelectorAll('button').forEach(b=>b.disabled=false);$('file').disabled=false;$('run').disabled=!chosen;}}
$('pause-live').onclick=async()=>{if(!currentJob)return;let action=paused?'resume':'pause';let response=await fetch('/api/jobs/'+currentJob+'/'+action,{method:'POST'});if(response.ok){paused=!paused;$('pause-live').textContent=paused?'Resume':'Pause';if(paused)$('status').textContent='Paused · tracker state preserved';}};
$('stop-live').onclick=()=>{if(currentJob){fetch('/api/jobs/'+currentJob+'/stop',{method:'POST'});$('stop-live').disabled=true;$('status').textContent='Stopping and saving recording…';}};
window.addEventListener('pagehide',()=>{if(currentJob)fetch('/api/jobs/'+currentJob+'/stop',{method:'POST',keepalive:true});});
</script><script src="/static/webcam.js"></script></body></html>


### static/webcam.js


In [ ]:
%%writefile /content/lavee_project/dashboard/static/webcam.js
let cameraStream=null,cameraSocket=null,cameraVideo=document.createElement('video');
cameraVideo.muted=true;cameraVideo.playsInline=true;
function stopCamera(){
 if(cameraSocket)cameraSocket.close();cameraSocket=null;
 if(cameraStream)cameraStream.getTracks().forEach(t=>t.stop());cameraStream=null;
 cameraVideo.srcObject=null;busy=false;
 document.querySelectorAll('button').forEach(b=>b.disabled=false);
 $('family').disabled=false;$('technique').disabled=false;$('file').disabled=false;
 $('camera-stop').hidden=true;$('run').disabled=!chosen;
}
$('camera-stop').onclick=()=>{stopCamera();$('status').textContent='Camera stopped';};
$('camera-start').onclick=async()=>{
 if(busy)return;
 clearResult();busy=true;
 document.querySelectorAll('button').forEach(b=>b.disabled=true);
 $('family').disabled=true;$('technique').disabled=true;$('file').disabled=true;
 try{
  cameraStream=await navigator.mediaDevices.getUserMedia({video:{width:{ideal:960},height:{ideal:540}},audio:false});
  cameraVideo.srcObject=cameraStream;await cameraVideo.play();
  const canvas=document.createElement('canvas'),ctx=canvas.getContext('2d');
  const ratio=Math.min(1,960/Math.max(cameraVideo.videoWidth,cameraVideo.videoHeight));
  canvas.width=Math.round(cameraVideo.videoWidth*ratio);canvas.height=Math.round(cameraVideo.videoHeight*ratio);
  const ws=cameraSocket=new WebSocket((location.protocol==='https:'?'wss://':'ws://')+location.host+'/api/webcam');
  function send(){if(cameraSocket!==ws||ws.readyState!==WebSocket.OPEN)return;ctx.drawImage(cameraVideo,0,0,canvas.width,canvas.height);canvas.toBlob(blob=>{if(blob&&ws.readyState===WebSocket.OPEN)ws.send(blob);},'image/jpeg',.8);}
  ws.onopen=()=>{ws.send(JSON.stringify({technique:selectedTechnique(),confidence:$('conf').value/100,iou:$('iou').value/100}));send();};
  ws.onmessage=e=>{let p=JSON.parse(e.data);if(p.type==='error'){showError(p.error);stopCamera();return;}drawLive(p);$('status').textContent='LIVE CAMERA · '+$('technique').selectedOptions[0].text+' · '+p.fps+' FPS';send();};
  ws.onerror=()=>{showError('Camera detection connection failed.');stopCamera();};
  ws.onclose=()=>{if(cameraSocket===ws)stopCamera();};
  $('camera-stop').hidden=false;$('camera-stop').disabled=false;
 }catch(e){stopCamera();showError('Cannot start camera: '+e.message);}
};
window.addEventListener('pagehide',stopCamera);


## 4. Load saved models and verify the two families


In [ ]:
import app as runtime
import classifiers,yolo12_backend
metadata=runtime.metadata()
assert metadata['families']['yolo12']['available'],'YOLO12 package is incomplete.'
for family,info in metadata['families'].items():print(family,info['name'],info['package'])
print('Class IDs:',metadata['classes'])
for key in classifiers.FILES:
    with classifiers.LOCK:classifiers.load(key)
    with yolo12_backend.LOCK:yolo12_backend.load(key)
print('All ten trained classifier heads loaded successfully.')


## 5. Select the detector and classification technique

Re-run this cell to change a notebook demo. The web dashboard has its own selectors.
`yolo` means baseline detection; all other methods add the corresponding trained crop classifier.


In [ ]:
MODEL_FAMILY='yolo12' # @param ['yolov8s','yolo12']
TECHNIQUE='bilstm_attention' # @param ['yolo','cnn','lstm','transformer','bilstm_attention','cgan_transformer']
CONFIDENCE=0.25 # @param {type:'slider',min:0.05,max:0.95,step:0.05}
IOU=0.45
assert MODEL_FAMILY in ('yolov8s','yolo12')
SELECTED=('yolo12_' if MODEL_FAMILY=='yolo12' else '')+TECHNIQUE
classifiers.validate(SELECTED)
print('Selected:',MODEL_FAMILY,TECHNIQUE)


## 6. Image detection — demo or upload

Set `UPLOAD_IMAGE=True` for your own image. Demo IDs: 0 airplane, 1 bird, 2 helicopter,
3 kite, 4 parachute, 5 drone. A demo's expected class does not guarantee a correct model prediction.


In [ ]:
import cv2
import pandas as pd
from IPython.display import display,Image,Video,HTML
UPLOAD_IMAGE=False
DEMO_CLASS=0
image_path=DASH/'demos'/f'{DEMO_CLASS}.jpg'
if UPLOAD_IMAGE:
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1,'Select one image.'
    image_path=Path(next(iter(uploaded))).resolve()
    del uploaded
image=cv2.imread(str(image_path))
assert image is not None,'Image cannot be decoded.'
result=runtime.detect(image,CONFIDENCE,IOU,SELECTED)
display(Image(data=base64.b64decode(result['image'].split(',',1)[1])))
display(pd.DataFrame(result['detections']))
(DASH/'results/notebook_image.json').write_text(json.dumps({k:v for k,v in result.items() if k!='image'},indent=2))


## 7. Continuous video inference and ByteTrack inside the notebook

Live annotated frames appear while inference runs; a recording is also saved afterwards.
Use a demo or upload one video up to 100 MB / two minutes. Stop this cell to request processing stop.
The mixed demo is a labeled four-camera mosaic; kite/parachute samples are still-image clips.
Inference FPS measures processing speed, not accuracy or the original video's frame rate.


In [ ]:
UPLOAD_VIDEO=False
CLIP_ID='class-0-1'
video_source=None
if UPLOAD_VIDEO:
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1,'Select one video.'
    video_source=Path(next(iter(uploaded))).resolve()
    assert video_source.stat().st_size<=100*1024*1024,'Video exceeds 100 MB.'
    del uploaded
else:
    library=json.loads((DASH/'demos/manifest.json').read_text())
    clip=next(c for item in library for c in item['clips'] if c['id']==CLIP_ID)
    video_source=DASH/'demos'/clip['file']
assert video_source.is_file()
job_id=uuid.uuid4().hex
runtime.jobs[job_id]={'status':'processing','progress':0}
task=runtime.pool.submit(runtime.process_video,job_id,video_source,CONFIDENCE,IOU,False,SELECTED)
frame_display=display(HTML('Starting tracking...'),display_id=True)
last_frame=0
try:
    while not task.done():
        packet=runtime.jobs[job_id].get('live')
        if packet and packet['frame']!=last_frame:
            frame_display.update(Image(data=base64.b64decode(packet['image'].split(',',1)[1])))
            last_frame=packet['frame']
        time.sleep(.1)
except KeyboardInterrupt:
    runtime.jobs[job_id]['stop']=True
    task.result(timeout=60)
task.result()
summary={k:v for k,v in runtime.jobs[job_id].items() if k!='live'}
print(summary)
if summary['status']=='complete':display(Video(str(DASH/'results'/f'{job_id}.mp4'),embed=True))
else:raise RuntimeError(summary.get('error','Video processing failed.'))


## 8. Show the actual recorded research results

Detector mAP and crop-classification F1 measure different tasks. YOLOv8s detector ten-fold results
were not provided. YOLO12 has detector and crop ten-fold results, but no ten-fold end-to-end result
or final locked-test evaluation. Image-level fallback folds can contain related source frames.


In [ ]:
research=(yolo12_backend.research_summary() if MODEL_FAMILY=='yolo12' else classifiers.research_summary())
print(research['scope'])
if 'detector' in research:
    display(pd.DataFrame([{'metric':k,'mean':v['mean'],'sample_std':v['std']} for k,v in research['detector'].items()]))
display(pd.DataFrame([{'method':k,**{m:v[m] for m in ['folds','cv_macro_f1','cv_std']},
    **({'test_accuracy':v['test_accuracy']} if 'test_accuracy' in v else {})} for k,v in research['methods'].items()]))


## 9. Open the existing two-mode web dashboard in Colab

This runs the same FastAPI dashboard, including both model selectors, upload and demo modes.
Colab supplies the browser proxy; the localhost URL on your own computer does not point to this runtime.
If Colab's proxy does not support the live WebSocket stream or camera permissions, use the live notebook
cell above for video, or run `python -m uvicorn app:app --host 127.0.0.1 --port 8765` locally from `dashboard/`.
No third-party tunnel or credential is configured here. Close the runtime when finished.


In [ ]:
import requests
PORT=8765
if 'dashboard_process' in globals() and dashboard_process.poll() is None:
    print("Reusing this notebook dashboard process.")
else:
    dashboard_log=open('/content/lavee_dashboard.log','w')
    dashboard_process=subprocess.Popen([sys.executable,'-m','uvicorn','app:app','--host','0.0.0.0','--port',str(PORT)],cwd=str(DASH),stdout=dashboard_log,stderr=subprocess.STDOUT)
for _ in range(60):
    if dashboard_process.poll() is not None:raise RuntimeError(Path('/content/lavee_dashboard.log').read_text())
    try:
        response=requests.get(f'http://127.0.0.1:{PORT}/api/model',timeout=2)
        response.raise_for_status()
        break
    except requests.RequestException:time.sleep(1)
else:raise RuntimeError('Dashboard did not start. Read /content/lavee_dashboard.log')
from google.colab import output
output.serve_kernel_port_as_window(PORT)


## 10. Optional: download the generated results


In [ ]:
export=Path('/content/LAVEE_Inference_Results.zip')
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted((DASH/'results').glob('*')):
        if path.is_file() and not path.name.endswith(('_raw.mp4','_upload.mp4')):z.write(path,path.name)
from google.colab import files
files.download(str(export))


## 11. Optional: stop the dashboard process


In [ ]:
if 'dashboard_process' in globals() and dashboard_process.poll() is None:
    dashboard_process.terminate()
    dashboard_process.wait(timeout=15)
    dashboard_log.close()
    print('Dashboard stopped.')
